[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-03-ray-cluster-setup.ipynb#scrollTo=d1e2f3c4)

---
# Day 3 · Ray Clusters — Local, Docker, and Kubernetes Setup
**certified-journeys / ray-certified** · Day 3 · Infrastructure

> **Goal for today:** Understand Ray's cluster architecture (head node, worker nodes, GCS), write production-ready cluster YAML configs for Docker and Kubernetes, and use `ray status` / `ray stop` to inspect and manage clusters.


In [ ]:
%pip install -q 'ray[default]' pyyaml


## Step 1 · Ray Cluster Architecture

A Ray cluster has two node types:

```
┌───────────────────────────────────────────────────┐
│                   Ray Cluster                     │
│                                                   │
│  ┌────────────────────────────────────────────┐   │
│  │              HEAD NODE                     │   │
│  │  ┌──────────┐  ┌──────────┐  ┌─────────┐  │   │
│  │  │  GCS     │  │ Scheduler│  │  Raylet │  │   │
│  │  │(metadata)│  │ (tasks)  │  │ (local) │  │   │
│  │  └──────────┘  └──────────┘  └─────────┘  │   │
│  │  ┌──────────────────────────────────────┐  │   │
│  │  │         Plasma Object Store          │  │   │
│  │  └──────────────────────────────────────┘  │   │
│  └────────────────────────────────────────────┘   │
│                                                   │
│  ┌──────────────┐  ┌──────────────┐               │
│  │  WORKER NODE │  │  WORKER NODE │  ...          │
│  │  ┌────────┐  │  │  ┌────────┐  │               │
│  │  │ Raylet │  │  │  │ Raylet │  │               │
│  │  └────────┘  │  │  └────────┘  │               │
│  │  ┌────────┐  │  │  ┌────────┐  │               │
│  │  │Plasma  │  │  │  │Plasma  │  │               │
│  │  └────────┘  │  │  └────────┘  │               │
│  └──────────────┘  └──────────────┘               │
└───────────────────────────────────────────────────┘
```

| Component | Lives on | Purpose |
|-----------|----------|---------|
| **GCS** (Global Control Store) | Head | Cluster-wide metadata: actors, resource availability, node registry |
| **Scheduler** | Head | Assigns tasks to Raylets based on resource availability |
| **Raylet** | Every node | Local scheduler + object manager for that node |
| **Plasma store** | Every node | Shared-memory object store; objects move between nodes when needed |
| **Worker processes** | Every node | Execute tasks and host actors |


In [ ]:
import ray
import yaml
import json
import subprocess
import textwrap

# Start a local cluster to demonstrate introspection APIs
ray.init(ignore_reinit_error=True)

# Inspect cluster nodes — equivalent to 'ray status' output
nodes = ray.nodes()
print(f"Cluster has {len(nodes)} node(s):")
for node in nodes:
    print(f"\n  Node ID      : {node['NodeID'][:16]}...")
    print(f"  Alive        : {node['Alive']}")
    print(f"  Node IP      : {node['NodeManagerAddress']}")
    print(f"  Resources    :")
    for res, qty in node['Resources'].items():
        print(f"    {res:<20} {qty}")

# Available resources across the cluster
print("\nCluster-wide available resources:")
for res, qty in ray.available_resources().items():
    print(f"  {res:<20} {qty}")


### What just happened?
- `ray.nodes()` returns the **node registry from the GCS** — the same data the `ray status` CLI command shows.
- Each node has a resource dictionary with CPU, memory, and optionally GPU counts.
- On a real multi-node cluster you would see multiple entries with different IP addresses.
- `ray.available_resources()` shows **currently free** resources (total minus what tasks are holding).


## Step 2 · `ray start` CLI — Forming a Cluster Manually

Before YAML configs, understand the raw CLI commands that form a cluster:

```bash
# On the HEAD node — starts GCS, Scheduler, and a local Raylet
ray start --head --port=6379 --dashboard-host=0.0.0.0

# On each WORKER node — joins the head node
ray start --address='<head-node-ip>:6379'

# Connect a Python driver to an existing cluster
ray.init(address='auto')   # finds the cluster via environment
ray.init(address='ray://<head-ip>:10001')  # explicit address

# Inspect and stop
ray status
ray stop
```

In production you use Docker or Kubernetes instead of running `ray start` manually — but understanding these commands helps debug cluster issues.


In [ ]:
# Simulate 'ray status' output programmatically
# (On a real cluster this would show real node states)

def format_ray_status(nodes: list, resources: dict) -> str:
    """Format cluster status like the 'ray status' CLI output."""
    lines = []
    lines.append("======== Cluster status: ========")
    lines.append(f"Node count: {len(nodes)}")
    lines.append("")

    # Separate head from workers
    head_nodes   = [n for n in nodes if n.get('RayletSocketName', '').endswith('-1')]
    worker_nodes = [n for n in nodes if n not in head_nodes]

    lines.append(f"Healthy:")
    for node in nodes:
        ip   = node.get('NodeManagerAddress', 'unknown')
        cpus = node.get('Resources', {}).get('CPU', 0)
        mem  = node.get('Resources', {}).get('memory', 0)
        lines.append(f"  1 node(s) with resources: CPU: {cpus:.0f} Memory: {mem/1e9:.2f} GiB  [{ip}]")

    lines.append("")
    lines.append("Resources:")
    lines.append("  Usage:")

    total   = {k: v for k, v in nodes[0].get('Resources', {}).items()}
    avail   = resources
    for res in ['CPU', 'memory', 'GPU']:
        if res in total:
            used = total[res] - avail.get(res, 0)
            if res == 'memory':
                lines.append(f"    {res}: {used/1e9:.2f}GiB / {total[res]/1e9:.2f}GiB")
            else:
                lines.append(f"    {res}: {used:.1f} / {total[res]:.1f}")

    return "\n".join(lines)

status_text = format_ray_status(ray.nodes(), ray.available_resources())
print(status_text)

# Show what 'ray stop' does (we won't actually stop it yet)
print("\n--- ray stop command ---")
print("ray stop                   # graceful — waits for tasks to finish")
print("ray stop --force           # immediate — kills all workers now")
print("ray stop --grace-period=30 # wait up to 30s then force")


### What just happened?
- We reproduced the `ray status` output format using the Python API — same data, different UI.
- **`ray stop`** sends SIGTERM to all Ray processes; `--force` sends SIGKILL if needed.
- On a multi-node cluster, `ray stop` on a **worker** node just removes that node; `ray stop` on the **head** tears down the entire cluster.
- Always run `ray stop` before shutting down nodes to allow graceful task checkpointing.


## Step 3 · Docker-Based Cluster YAML Config

Ray's cluster launcher (`ray up`) uses a YAML config file to provision and configure nodes. For Docker-based deployments, you define:
- **`cluster_name`** — logical name used in all Ray logs
- **`docker`** — image, container name, run options
- **`head_node_type`** / **`worker_node_types`** — resource specs per node type
- **`setup_commands`** — shell commands run once on each node after provisioning
- **`head_start_ray_commands`** / **`worker_start_ray_commands`** — how Ray starts

The `ray up cluster.yaml` command provisions all nodes and starts Ray — one command to launch a cluster.


In [ ]:
# Generate a Docker-based Ray cluster config (1 head + 2 workers)

docker_cluster_config = {
    "cluster_name": "my-ray-cluster",

    # Maximum number of worker nodes the autoscaler can add
    "max_workers": 4,

    # Upscale/downscale aggressiveness (seconds)
    "upscaling_speed": 1.0,
    "idle_timeout_minutes": 5,

    # Docker settings — applied to ALL nodes
    "docker": {
        "image": "rayproject/ray:2.9.0-py310",   # official Ray Docker image
        "container_name": "ray_container",
        "pull_before_run": True,
        "run_options": [
            "--ulimit nofile=65536:65536",         # needed for large clusters
            "--shm-size=2gb",                      # Plasma object store uses /dev/shm
        ]
    },

    # Provider — 'local' for Docker Compose / bare metal
    # In production: 'aws', 'gcp', 'azure'
    "provider": {
        "type": "local",
        "head_ip": "127.0.0.1",           # head node IP
        "worker_ips": [                    # static worker IPs
            "192.168.1.101",
            "192.168.1.102",
        ]
    },

    # SSH credentials — used by ray up to connect and configure nodes
    "auth": {
        "ssh_user": "ubuntu",
        "ssh_private_key": "~/.ssh/ray_cluster_key",
    },

    # Available node types (head node type + worker types)
    "available_node_types": {
        "ray.head.default": {
            "resources": {},              # head resources auto-detected
            "node_config": {
                "instance_type": "head"
            }
        },
        "ray.worker.default": {
            "min_workers": 2,             # always keep 2 workers
            "max_workers": 4,             # scale up to 4 when busy
            "resources": {
                "CPU": 4,
                "memory": 8_000_000_000,  # 8 GB
            },
            "node_config": {
                "instance_type": "worker"
            }
        }
    },

    "head_node_type": "ray.head.default",

    # Commands run on EVERY node after Docker container starts
    "setup_commands": [
        "pip install 'ray[default]' numpy pandas",
    ],

    # Commands that start Ray on the head node
    "head_start_ray_commands": [
        "ray stop",
        "ray start --head --port=6379 --dashboard-host=0.0.0.0 --block",
    ],

    # Commands that start Ray on each worker node
    "worker_start_ray_commands": [
        "ray stop",
        "ray start --address=$RAY_HEAD_IP:6379 --block",
    ],
}

yaml_output = yaml.dump(docker_cluster_config, default_flow_style=False, sort_keys=False, allow_unicode=True)
print("=== Docker Ray Cluster Config (cluster.yaml) ===")
print(yaml_output)


### What just happened?
- We generated a complete `cluster.yaml` for a 1 head + 2–4 worker Docker cluster.
- **`--shm-size=2gb`** in Docker run options allocates shared memory for the Plasma object store — this is a common gotcha; too small and Ray objects get evicted.
- `min_workers: 2` ensures 2 workers are always running; autoscaler adds more up to `max_workers: 4`.
- Run `ray up cluster.yaml` to launch the cluster; `ray down cluster.yaml` to tear it down.


## Step 4 · KubeRay — RayCluster Custom Resource Definition

On Kubernetes, you don't run `ray start` manually. Instead, **KubeRay** (the Ray Kubernetes Operator) manages Ray clusters via a `RayCluster` Custom Resource Definition (CRD). You describe the desired cluster state in YAML and `kubectl apply` it — Kubernetes handles the rest.

```bash
# Install KubeRay operator (Helm)
helm repo add kuberay https://ray-project.github.io/kuberay-helm/
helm install kuberay-operator kuberay/kuberay-operator

# Apply the RayCluster manifest
kubectl apply -f raycluster.yaml

# Inspect
kubectl get raycluster
kubectl get pods -l ray.io/cluster=my-ray-cluster

# Port-forward the dashboard
kubectl port-forward svc/my-ray-cluster-head-svc 8265:8265
```


In [ ]:
# Generate a production-ready KubeRay RayCluster manifest

kuberay_cluster = {
    "apiVersion": "ray.io/v1",
    "kind": "RayCluster",
    "metadata": {
        "name": "my-ray-cluster",
        "namespace": "ray-system",
        "labels": {
            "app": "ray",
            "environment": "production"
        }
    },
    "spec": {
        "rayVersion": "2.9.0",
        "enableInTreeAutoscaling": True,
        "autoscalerOptions": {
            "upscalingMode": "Default",
            "idleTimeoutSeconds": 60,
            "resources": {
                "requests": {"cpu": "100m", "memory": "128Mi"},
                "limits":   {"cpu": "500m", "memory": "512Mi"},
            }
        },
        # HEAD NODE GROUP
        "headGroupSpec": {
            "serviceType": "ClusterIP",
            "rayStartParams": {
                "dashboard-host":     "0.0.0.0",
                "num-cpus":           "0",     # head should not run user tasks
                "block":              "true",
            },
            "template": {
                "metadata": {"labels": {"ray.io/node-type": "head"}},
                "spec": {
                    "containers": [{
                        "name":  "ray-head",
                        "image": "rayproject/ray:2.9.0-py310",
                        "ports": [
                            {"name": "gcs",       "containerPort": 6379},
                            {"name": "dashboard", "containerPort": 8265},
                            {"name": "client",    "containerPort": 10001},
                        ],
                        "resources": {
                            "requests": {"cpu": "1",    "memory": "2Gi"},
                            "limits":   {"cpu": "2",    "memory": "4Gi"},
                        },
                        "env": [
                            {"name": "RAY_DISABLE_IMPORT_WARNING", "value": "1"}
                        ],
                        # Plasma uses /dev/shm — give it enough space
                        "volumeMounts": [{
                            "mountPath": "/dev/shm",
                            "name": "dshm"
                        }]
                    }],
                    "volumes": [{
                        "name": "dshm",
                        "emptyDir": {"medium": "Memory", "sizeLimit": "2Gi"}
                    }]
                }
            }
        },
        # WORKER NODE GROUPS
        "workerGroupSpecs": [
            {
                "replicas":    2,
                "minReplicas": 2,
                "maxReplicas": 8,
                "groupName":   "cpu-workers",
                "rayStartParams": {
                    "num-cpus": "4",
                    "block":    "true",
                },
                "template": {
                    "metadata": {"labels": {"ray.io/node-type": "worker"}},
                    "spec": {
                        "containers": [{
                            "name":  "ray-worker",
                            "image": "rayproject/ray:2.9.0-py310",
                            "resources": {
                                "requests": {"cpu": "4",    "memory": "8Gi"},
                                "limits":   {"cpu": "4",    "memory": "8Gi"},
                            },
                            "lifecycle": {
                                "preStop": {"exec": {"command": ["/bin/sh", "-c", "ray stop"]}}
                            },
                            "volumeMounts": [{
                                "mountPath": "/dev/shm",
                                "name": "dshm"
                            }]
                        }],
                        "volumes": [{
                            "name": "dshm",
                            "emptyDir": {"medium": "Memory", "sizeLimit": "4Gi"}
                        }]
                    }
                }
            }
        ]
    }
}

kuberay_yaml = yaml.dump(kuberay_cluster, default_flow_style=False, sort_keys=False, allow_unicode=True)
print("=== KubeRay RayCluster Manifest ===")
print(kuberay_yaml)


### What just happened?
- We generated a complete `RayCluster` CRD manifest — `kubectl apply -f` this file to launch the cluster.
- **`num-cpus: "0"` on the head** prevents user tasks from landing on the head node (it should only run GCS/Scheduler).
- The `emptyDir: {medium: Memory}` volume mounts `/dev/shm` with bounded size — critical for Plasma on Kubernetes.
- **`lifecycle.preStop`** runs `ray stop` before a pod terminates — enables graceful draining of in-flight tasks.


## Step 5 · Autoscaling — How Ray Scales Worker Nodes

Ray's autoscaler monitors **pending task queue depth** and **resource demands**. It adds nodes when tasks are waiting for resources and removes idle nodes after `idle_timeout_minutes`.

| Signal | Autoscaler action |
|--------|------------------|
| Tasks waiting for CPU/GPU that the cluster doesn't have | Add worker nodes |
| Nodes idle > `idle_timeout_minutes` | Remove worker nodes |
| Node fails health check | Replace node |
| `max_workers` reached | Queue tasks (no more nodes added) |

Key autoscaling knobs:
- **`min_workers`** — always keep this many workers (prevents cold-start latency)
- **`max_workers`** — hard cap on total worker nodes
- **`upscaling_speed`** — aggressiveness of scale-out (1.0 = default)
- **`idle_timeout_minutes`** — how long before idle nodes are released


In [ ]:
# Simulate autoscaler decision logic
# (In production this runs inside the Ray head node process)

import time
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class ClusterState:
    """Simplified model of Ray autoscaler state."""
    min_workers:          int   = 2
    max_workers:          int   = 8
    cpus_per_worker:      int   = 4
    idle_timeout_minutes: float = 5.0
    current_workers:      int   = 2      # start at min
    pending_cpu_demand:   int   = 0      # CPUs needed by queued tasks
    idle_worker_minutes:  Dict[int, float] = field(default_factory=dict)

    @property
    def total_cpus(self) -> int:
        return self.current_workers * self.cpus_per_worker

    def autoscale(self, pending_cpus: int, idle_minutes: float = 0) -> str:
        """Decide whether to scale up, down, or hold."""
        self.pending_cpu_demand = pending_cpus

        # Scale-up: add workers to meet pending demand
        if pending_cpus > 0:
            workers_needed = (pending_cpus + self.cpus_per_worker - 1) // self.cpus_per_worker
            target = min(self.current_workers + workers_needed, self.max_workers)
            if target > self.current_workers:
                added = target - self.current_workers
                self.current_workers = target
                return f"SCALE UP   +{added} workers → {self.current_workers} total ({self.total_cpus} CPUs)"

        # Scale-down: remove workers idle longer than timeout
        if idle_minutes >= self.idle_timeout_minutes and self.current_workers > self.min_workers:
            self.current_workers -= 1
            return f"SCALE DOWN -1 worker  → {self.current_workers} total ({self.total_cpus} CPUs)"

        return f"NO CHANGE             → {self.current_workers} total ({self.total_cpus} CPUs)"


# Simulate a workload scenario
cluster = ClusterState(min_workers=2, max_workers=8, cpus_per_worker=4)
print(f"Initial state: {cluster.current_workers} workers, {cluster.total_cpus} CPUs")
print()

scenario = [
    # (time_label,        pending_cpus, idle_minutes)
    ("T+0:  Workload arrives",   20, 0.0),
    ("T+1:  More tasks queued",  12, 0.0),
    ("T+2:  Workload draining",   0, 0.0),
    ("T+7:  Workers idle 5 min",  0, 5.0),
    ("T+8:  Still idle",          0, 5.0),
    ("T+9:  Back to min_workers", 0, 5.0),
]

for label, pending, idle in scenario:
    decision = cluster.autoscale(pending_cpus=pending, idle_minutes=idle)
    print(f"  {label:<32} pending_cpus={pending:>2}  {decision}")


### What just happened?
- The autoscaler **scaled up** to meet demand (20 pending CPUs → added workers) then **scaled down** after 5 minutes of idle.
- **`min_workers`** acts as a floor — the cluster never drops below 2 workers regardless of idleness.
- This logic runs as a loop in the Ray head node — you configure it via cluster YAML, not code.
- For Kubernetes, KubeRay translates scale-up/down decisions into `kubectl scale` calls on the worker ReplicaSet.


## Step 6 · `ray.init()` Connection Modes

Understanding when to use each `ray.init()` mode is essential for debugging cluster issues:

| Mode | Code | When to use |
|------|------|-------------|
| Local cluster | `ray.init()` | Development; auto-starts cluster using local CPUs |
| Connect to existing | `ray.init(address='auto')` | Head node started with `ray start --head` |
| Explicit address | `ray.init(address='ray://1.2.3.4:10001')` | Remote cluster, Ray Client port |
| Config override | `ray.init(num_cpus=2, num_gpus=1)` | Override auto-detected resources |
| Silent init | `ray.init(logging_level='error')` | Suppress startup messages in notebooks |


In [ ]:
# Demonstrate introspection of the current cluster
# (Ray is already running from the earlier init)

def cluster_summary() -> dict:
    """Return a dict summarising the current Ray cluster."""
    ctx = ray.get_runtime_context()
    nodes = ray.nodes()
    alive_nodes = [n for n in nodes if n['Alive']]

    total_res  = {}
    for node in alive_nodes:
        for res, qty in node.get('Resources', {}).items():
            total_res[res] = total_res.get(res, 0) + qty

    avail_res = ray.available_resources()

    return {
        "dashboard_url":  ray.get_dashboard_url() or "http://localhost:8265",
        "gcs_address":    ctx.gcs_address if hasattr(ctx, 'gcs_address') else "localhost:6379",
        "alive_nodes":    len(alive_nodes),
        "total_resources": {
            k: f"{v:.0f}" if k != 'memory' else f"{v/1e9:.2f} GiB"
            for k, v in total_res.items()
        },
        "available_resources": {
            k: f"{v:.0f}" if k != 'memory' else f"{v/1e9:.2f} GiB"
            for k, v in avail_res.items()
        },
    }

summary = cluster_summary()
print("=== Current Cluster Summary ===")
for key, val in summary.items():
    if isinstance(val, dict):
        print(f"  {key}:")
        for k, v in val.items():
            print(f"    {k:<20} {v}")
    else:
        print(f"  {key:<22} {val}")

# Show how to connect to a remote cluster (not executed, just printed)
print("\n=== Connection recipes ===")
recipes = [
    ("Local dev",            "ray.init()"),
    ("Auto-detect",          "ray.init(address='auto')"),
    ("Remote Ray Client",    "ray.init(address='ray://head-ip:10001')"),
    ("KubeRay port-forward", "ray.init(address='ray://localhost:10001')"),
    ("Limit local CPUs",     "ray.init(num_cpus=4)"),
]
for label, code in recipes:
    print(f"  {label:<25} {code}")


### What just happened?
- `ray.get_dashboard_url()` returns the URL of the Ray Dashboard — a web UI for monitoring tasks, actors, and resources.
- `ray.get_runtime_context()` gives you the current job's metadata — job ID, task ID, actor ID.
- **The GCS address** is what worker nodes use to register with the head node; keep port 6379 open between nodes.
- For KubeRay, use `kubectl port-forward` to access the dashboard and Ray Client from your laptop.


## Step 7 · Docker Compose Cluster — Local Multi-Node Simulation

For local multi-node testing without a full Kubernetes setup, Docker Compose lets you simulate a real Ray cluster on a single machine with network isolation between containers.


In [ ]:
# Generate a Docker Compose file for a local Ray cluster (1 head + 2 workers)

ray_image = "rayproject/ray:2.9.0-py310"

docker_compose = {
    "version": "3.8",
    "networks": {
        "ray-network": {"driver": "bridge"}
    },
    "services": {
        # HEAD NODE — runs GCS, Scheduler, Dashboard
        "ray-head": {
            "image": ray_image,
            "container_name": "ray-head",
            "command": [
                "ray", "start", "--head",
                "--port=6379",
                "--dashboard-host=0.0.0.0",
                "--dashboard-port=8265",
                "--num-cpus=2",
                "--block",
            ],
            "ports": [
                "6379:6379",    # GCS / cluster port
                "8265:8265",    # Ray Dashboard
                "10001:10001",  # Ray Client (for remote drivers)
            ],
            "shm_size": "2gb",   # Plasma object store
            "networks": ["ray-network"],
            "healthcheck": {
                "test": ["CMD", "ray", "status"],
                "interval": "10s",
                "timeout":  "5s",
                "retries":  10,
            }
        },
        # WORKER NODE 1
        "ray-worker-1": {
            "image": ray_image,
            "container_name": "ray-worker-1",
            "command": [
                "ray", "start",
                "--address=ray-head:6379",  # connect to head via Docker DNS
                "--num-cpus=4",
                "--block",
            ],
            "shm_size": "4gb",
            "networks": ["ray-network"],
            "depends_on": {"ray-head": {"condition": "service_healthy"}},
        },
        # WORKER NODE 2
        "ray-worker-2": {
            "image": ray_image,
            "container_name": "ray-worker-2",
            "command": [
                "ray", "start",
                "--address=ray-head:6379",
                "--num-cpus=4",
                "--block",
            ],
            "shm_size": "4gb",
            "networks": ["ray-network"],
            "depends_on": {"ray-head": {"condition": "service_healthy"}},
        },
    }
}

compose_yaml = yaml.dump(docker_compose, default_flow_style=False, sort_keys=False, allow_unicode=True)
print("=== Docker Compose Ray Cluster ===")
print(compose_yaml)

print("Usage:")
print("  docker-compose up -d                  # start the cluster")
print("  docker-compose logs -f ray-head        # tail head node logs")
print("  docker exec -it ray-head ray status    # inspect cluster inside container")
print("  docker-compose down                    # stop and remove containers")


### What just happened?
- The Docker Compose file creates a **private bridge network** (`ray-network`) so containers resolve each other by service name (`ray-head`, `ray-worker-1`).
- Workers connect via `--address=ray-head:6379` — Docker's internal DNS resolves `ray-head` to the container IP.
- `depends_on: service_healthy` ensures workers only start after the head's `ray status` health check passes.
- Connect a Python driver from your laptop: `ray.init(address='ray://localhost:10001')`.


In [ ]:
# Challenge: Generate a custom KubeRay manifest
#
# Create a function `make_kuberay_manifest(name, namespace, ray_version,
#   head_cpu, head_memory_gi, worker_replicas, worker_cpu, worker_memory_gi,
#   min_replicas, max_replicas)` that returns a dict representing
# a valid KubeRay RayCluster manifest.
#
# Requirements:
#   - apiVersion: ray.io/v1, kind: RayCluster
#   - Head node: num-cpus="0" (no user tasks on head)
#   - Worker group: use the provided replica counts and resource values
#   - Include shm volume mounts on both head and workers (4Gi for workers)
#   - enableInTreeAutoscaling: True
#
# Then call the function and print the result as YAML for a cluster named
# 'prod-cluster' in namespace 'ml-platform', Ray 2.9.0,
# head 2CPU/4Gi, 3 workers (min=3, max=10) each 8CPU/32Gi.

# Your solution here:
# def make_kuberay_manifest(name, namespace, ray_version,
#                           head_cpu, head_memory_gi,
#                           worker_replicas, worker_cpu, worker_memory_gi,
#                           min_replicas, max_replicas) -> dict:
#     ...

# manifest = make_kuberay_manifest(...)
# print(yaml.dump(manifest, default_flow_style=False, sort_keys=False))


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Head node | Runs GCS + Scheduler; should NOT run user tasks (`num-cpus=0`) |
| Worker node | Runs Raylets + Plasma store; executes tasks and actors |
| GCS | Global Control Store — cluster-wide metadata (actors, resource registry) |
| `ray start --head` | Starts head node; workers join with `--address=head-ip:6379` |
| `ray.init(address='auto')` | Connects driver to an already-running cluster |
| `ray status` | CLI view of nodes, resources, and tasks from GCS data |
| `ray up cluster.yaml` | Provisions + starts a cluster from a YAML spec (Docker or cloud) |
| KubeRay CRD | `RayCluster` resource manages Ray on Kubernetes via the operator |
| Autoscaler | Adds workers when tasks are pending; removes idle workers after timeout |
| `shm_size` / `/dev/shm` | Critical for Plasma — set large enough for your object store usage |

> **Tip:** For local development, `ray.init()` without arguments spins up a local cluster automatically. Use `ray.init(address='auto')` only when connecting to a pre-existing cluster started with `ray start --head`.

---
## What's next
**Day 4** → Ray Data — scalable dataset ingestion, preprocessing, and streaming with Ray's distributed Dataset API.

Mark Day 3 complete in your [tracker](../index.html).


In [ ]:
# Shut down Ray cleanly at the end of the notebook
ray.shutdown()
print("Ray cluster shut down.")
